# LSH + MinHash vs REACTA Paper Baselines
**Full Dataset Scale — Paper Evaluation Protocol**

This notebook matches the scale and evaluation protocol of the REACTA paper (Tran et al., RecSys 2025 / arxiv 2507.17356):
- **Data**: All 500 session files (~893M events, 4M+ users, 50k tracks)
- **Users**: Only those with ≥31 sessions (need L=30 input + 1 target, matching paper)
- **Evaluation**: Sliding window — observe last 30 sessions, predict tracks in session 31
- **Metrics**: Recall@10 and NDCG@10 reported as % to match paper Table 1

**Two approximate algorithms from class:**
- **Component A — LSH (Random Projection)**: Hash track SVD embeddings → retrieve ~hundreds of candidates instead of 50k
- **Component B — MinHash CF**: Approximate Jaccard similarity between users → collaborative filtering without O(N²) comparison

---
# PART A: LSH Candidate Generation

In [1]:
import numpy as np
import pandas as pd
import pickle
import time
import gc
import os
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH  = "/Users/spartan/Desktop/Music Recommendation/deezer-recsys25"
CACHE_PATH = "/Users/spartan/Desktop/Music Recommendation/Music-Recommendation/cache"
os.makedirs(CACHE_PATH, exist_ok=True)

# ── Paper evaluation protocol (REACTA RecSys 2025) ──────────────────────────
L_INPUT      = 30     # sessions used as context (paper: L=30)
K_TRACKS     = 10     # top-K for all metrics
N_TEST_WIN   = 10     # test windows per user (paper: last 10 sequences for test)
MIN_SESSIONS = L_INPUT + 1   # ≥31 sessions required per user

# Cap evaluation queries for tractable runtime (sample across many users)
MAX_EVAL_QUERIES = 10_000

# Paper baselines — Table 1, REACTA paper
PAPER_BASELINES = {
    "ACT-R-Repeat": {"Recall@10": 11.70, "NDCG@10": 10.74},
    "ACT-R-BPR":    {"Recall@10":  7.26, "NDCG@10":  7.41},
    "PISA-U":       {"Recall@10":  8.39, "NDCG@10":  8.34},
    "PISA-P":       {"Recall@10":  8.86, "NDCG@10":  8.88},
    "REACTA-U":     {"Recall@10":  8.45, "NDCG@10":  8.56},
    "REACTA-P":     {"Recall@10":  9.10, "NDCG@10":  9.35},   # best
}

np.random.seed(42)
print(f"Paper protocol: observe {L_INPUT} sessions → predict session {L_INPUT+1}")
print(f"Users need ≥{MIN_SESSIONS} sessions | eval cap: {MAX_EVAL_QUERIES:,} queries")
print(f"Paper best baseline  →  REACTA-P: Recall@10={PAPER_BASELINES['REACTA-P']['Recall@10']}%  "
      f"NDCG@10={PAPER_BASELINES['REACTA-P']['NDCG@10']}%")

Paper protocol: observe 30 sessions → predict session 31
Users need ≥31 sessions | eval cap: 10,000 queries
Paper best baseline  →  REACTA-P: Recall@10=9.1%  NDCG@10=9.35%


## A1 — Load Track SVD Embeddings

In [2]:
print("Loading track SVD embeddings...")
t0 = time.time()

emb_files = sorted(Path(f"{DATA_PATH}/track_embeddings").glob("svd_audio_*"))
raw = pd.concat([pd.read_parquet(f) for f in emb_files], ignore_index=True)

def extract_svd(x):
    if isinstance(x, dict) and 'list' in x:
        return np.array([item['item'] for item in x['list'][:128]], dtype=np.float32)
    return np.array(x, dtype=np.float32)[:128]

track_ids  = raw['track_id'].tolist()
embeddings = np.stack(raw['svd'].apply(extract_svd).values)   # (50000, 128)

# L2-normalise: cosine similarity becomes dot product, required for random-projection LSH
norms      = np.linalg.norm(embeddings, axis=1, keepdims=True)
emb_norm   = embeddings / (norms + 1e-8)                      # (50000, 128)

# Fast lookup: track_id → normalised embedding
svd_lookup = {tid: emb_norm[i] for i, tid in enumerate(track_ids)}
tid_array  = np.array(track_ids)                              # for vectorised ops

del raw; gc.collect()
print(f"Loaded {len(track_ids):,} tracks  shape={emb_norm.shape}  ({time.time()-t0:.1f}s)")

Loading track SVD embeddings...
Loaded 50,000 tracks  shape=(50000, 128)  (2.2s)


## A2 — Random Projection LSH Index

**Theory**: For cosine similarity, a random hyperplane through the origin splits the space. Two vectors on the same side get the same bit. The probability they agree on one bit equals `1 - θ/π` where θ is the angle between them. With `b` bits per table, bucket collision probability is `(1 - θ/π)^b`. With `L` independent tables, a pair is retrieved if it matches in at least one table.

**Parameters**: L tables × b bits per table. More tables → higher recall, larger candidate set.

In [3]:
class RandomProjectionLSH:
    """
    LSH for cosine similarity using random hyperplanes.
    Each table hashes a vector to a b-bit code (one bit per hyperplane).
    Items landing in the same bucket across any table become candidates.
    """
    def __init__(self, dim: int, n_tables: int = 10, n_bits: int = 8, seed: int = 42):
        rng = np.random.RandomState(seed)
        # planes[t]: (n_bits, dim) — random unit hyperplanes for table t
        self.planes   = [rng.randn(n_bits, dim).astype(np.float32) for _ in range(n_tables)]
        self.n_tables = n_tables
        self.n_bits   = n_bits
        self.tables   = [defaultdict(list) for _ in range(n_tables)]  # bucket → [indices]
        self.track_ids = []

    def _hash(self, vec: np.ndarray, plane: np.ndarray) -> tuple:
        """Project vec onto each hyperplane, take sign → b-bit hash code."""
        return tuple((plane @ vec > 0).astype(np.int8).tolist())

    def build(self, track_ids: list, emb_norm: np.ndarray):
        """Index all tracks. emb_norm must be L2-normalised."""
        self.track_ids = track_ids
        t0 = time.time()
        for idx in range(len(track_ids)):
            vec = emb_norm[idx]
            for t, plane in enumerate(self.planes):
                self.tables[t][self._hash(vec, plane)].append(idx)
        print(f"Built LSH index: {len(track_ids):,} tracks, "
              f"{self.n_tables} tables × {self.n_bits} bits  ({time.time()-t0:.1f}s)")

    def query(self, query_vec: np.ndarray, n_tables: int = None) -> list:
        """Return candidate track_ids from matching buckets across n_tables tables."""
        n = n_tables or self.n_tables
        candidate_idx = set()
        for t in range(n):
            h = self._hash(query_vec, self.planes[t])
            candidate_idx.update(self.tables[t].get(h, []))
        return [self.track_ids[i] for i in candidate_idx]

    def stats(self):
        sizes = [len(v) for table in self.tables for v in table.values()]
        print(f"Bucket stats — mean: {np.mean(sizes):.1f}  "
              f"median: {np.median(sizes):.0f}  max: {np.max(sizes)}")

print("LSH class defined.")

LSH class defined.


## A3 — Build Index

In [4]:
# N_TABLES=10, N_BITS=8 → 256 possible buckets per table
# We'll sweep N_TABLES from 1..10 during evaluation, so build with max=10
N_TABLES = 10
N_BITS   = 8

lsh = RandomProjectionLSH(dim=128, n_tables=N_TABLES, n_bits=N_BITS)
lsh.build(track_ids, emb_norm)
lsh.stats()

# Quick sanity check
sample_tid  = track_ids[0]
sample_q    = svd_lookup[sample_tid]
candidates  = lsh.query(sample_q)
print(f"\nSanity: query track {sample_tid} → {len(candidates)} candidates "
      f"(out of {len(track_ids):,} total)")
print(f"Query track in its own candidates: {sample_tid in candidates}")

Built LSH index: 50,000 tracks, 10 tables × 8 bits  (1.1s)
Bucket stats — mean: 195.3  median: 105  max: 3169

Sanity: query track 25039 → 6233 candidates (out of 50,000 total)
Query track in its own candidates: True


## A4 — Full-Scale Data Loading (All 500 Session Files)

Two-pass loading strategy to handle the full dataset:
- **Pass 1** — read only `user_id` + `session_id` from all 500 files to count sessions per user (~5 min). Identifies the subset with ≥31 sessions.
- **Pass 2** — reload full columns (`user_id`, `session_id`, `track_id`, `ts`) for qualifying users only. Much smaller since we keep only heavy users.
- Results cached to disk — subsequent runs load instantly.

In [5]:
SESS_CACHE = f"{CACHE_PATH}/sessions_full_L{MIN_SESSIONS}.pkl"

# Remove stale empty cache if it exists
if Path(SESS_CACHE).exists():
    import os as _os
    cached = pickle.load(open(SESS_CACHE, 'rb'))
    if len(cached) == 0:
        _os.remove(SESS_CACHE)
        print(f"Removed empty cache, will rebuild.")
        del cached

if Path(SESS_CACHE).exists():
    print(f"Cache found — loading {SESS_CACHE}")
    t0 = time.time()
    with open(SESS_CACHE, 'rb') as f:
        user_sessions = pickle.load(f)
    print(f"Loaded {len(user_sessions):,} users  ({time.time()-t0:.1f}s)")

else:
    # Files are parquet but have NO .parquet extension
    session_files = sorted(Path(f"{DATA_PATH}/user_sessions").glob("sessions_*"))
    print(f"No cache found. Processing {len(session_files)} session files ...")

    # ── Pass 1: count unique sessions per user ────────────────────────────
    print(f"\nPass 1 / 2: counting sessions (columns: user_id, session_id only) ...")
    t1 = time.time()
    sess_sets = defaultdict(set)   # uid → set of session_ids

    for i, fpath in enumerate(session_files):
        df = pd.read_parquet(fpath, columns=['user_id', 'session_id'])
        for uid, sid in zip(df['user_id'].to_numpy(), df['session_id'].to_numpy()):
            sess_sets[uid].add(sid)
        del df
        if (i + 1) % 50 == 0:
            print(f"  {i+1:3d}/{len(session_files)}  users_seen={len(sess_sets):,}", end='\r')

    qualifying = {uid for uid, sids in sess_sets.items() if len(sids) >= MIN_SESSIONS}
    del sess_sets; gc.collect()
    print(f"\nPass 1 done ({time.time()-t1:.1f}s): "
          f"{len(qualifying):,} users with ≥{MIN_SESSIONS} sessions")

    # ── Pass 2: load full events for qualifying users ─────────────────────
    print(f"\nPass 2 / 2: loading full events for {len(qualifying):,} users ...")
    t2 = time.time()
    events = defaultdict(lambda: defaultdict(list))   # uid → sid → [(ts, tid)]

    for i, fpath in enumerate(session_files):
        df = pd.read_parquet(fpath, columns=['user_id', 'session_id', 'track_id', 'ts'])
        df = df[df['user_id'].isin(qualifying)]
        if len(df):
            for row in df.itertuples(index=False):
                events[row.user_id][row.session_id].append((row.ts, row.track_id))
        del df
        if (i + 1) % 50 == 0:
            print(f"  {i+1:3d}/{len(session_files)}", end='\r')

    # Sort sessions by their earliest timestamp → ordered session list per user
    print("\nSorting sessions by timestamp ...")
    user_sessions = {}
    for uid, sess_dict in events.items():
        sorted_sids = sorted(sess_dict,
                             key=lambda sid: sess_dict[sid][0][0] if sess_dict[sid] else 0)
        session_list = []
        for sid in sorted_sids:
            tracks = [tid for _, tid in sorted(sess_dict[sid])]
            if tracks:
                session_list.append(tracks)
        if len(session_list) >= MIN_SESSIONS:
            user_sessions[uid] = session_list

    del events; gc.collect()
    print(f"Final: {len(user_sessions):,} users with ≥{MIN_SESSIONS} ordered sessions")

    print(f"\nSaving to cache: {SESS_CACHE}")
    with open(SESS_CACHE, 'wb') as f:
        pickle.dump(user_sessions, f)
    print(f"Total load time: {time.time()-t1:.1f}s")

# Dataset stats
n_sess   = [len(v) for v in user_sessions.values()]
all_trks = [t for sessions in user_sessions.values() for sess in sessions for t in sess]
print(f"\nUsers: {len(user_sessions):,}")
print(f"Sessions: median={np.median(n_sess):.0f}  mean={np.mean(n_sess):.1f}  max={max(n_sess)}")
print(f"Unique tracks in sessions: {len(set(all_trks)):,}")
del all_trks

No cache found. Processing 500 session files ...

Pass 1 / 2: counting sessions (columns: user_id, session_id only) ...


KeyboardInterrupt: 

## A4b — Build Evaluation Queries (Paper Evaluation Protocol)

For each qualifying user, create sliding windows of L+1 consecutive sessions:
- **Input**: sessions `[i : i+30]` — user's last 30 sessions as context
- **Target**: `sessions[i+30]` — tracks in the next session (what we want to predict)
- Use the last `N_TEST_WIN=10` such windows per user as test queries

This mirrors the paper exactly: a model that sees the last 30 sessions must predict the 31st.

In [ ]:
# ── Candidate universe: tracks with SVD embeddings that appear in sessions ───
session_tracks = set(
    t for sessions in user_sessions.values()
    for sess in sessions for t in sess
    if t in svd_lookup
)
UNIVERSE_SIZE = len(session_tracks)
print(f"Candidate universe (tracks with embeddings): {UNIVERSE_SIZE:,}")

# ── Helper metric functions (paper-style) ────────────────────────────────────
def recall_at_k(recs, gt_set, k=10):
    """Fraction of ground-truth tracks found in top-k recommendations."""
    hits = sum(1 for t in recs[:k] if t in gt_set)
    return hits / len(gt_set) if gt_set else 0.0

def ndcg_at_k(recs, gt_set, k=10):
    """NDCG@k — each GT track has relevance=1, penalises lower ranks."""
    dcg  = sum(1.0 / np.log2(i + 2) for i, t in enumerate(recs[:k]) if t in gt_set)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def cosine_topk(qvec, candidate_tids, k=10):
    """Return top-k candidates ranked by dot product (embeddings are L2-normalised)."""
    scores = {t: float(np.dot(qvec, svd_lookup[t])) for t in candidate_tids if t in svd_lookup}
    return sorted(scores, key=scores.get, reverse=True)[:k]

def context_vec(sessions_slice):
    """Mean L2-normalised SVD of all tracks in a list of sessions."""
    vecs = [svd_lookup[t] for sess in sessions_slice for t in sess if t in svd_lookup]
    if not vecs:
        return None
    v = np.mean(vecs, axis=0).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

# ── Build sliding-window evaluation queries ───────────────────────────────────
print("\nBuilding evaluation queries ...")
t0 = time.time()

eval_queries = []   # (uid, query_vec, gt_set, context_sessions)
uid_order = list(user_sessions.keys())
np.random.shuffle(uid_order)   # random user order for unbiased sampling

for uid in uid_order:
    sessions = user_sessions[uid]
    n = len(sessions)
    # Last N_TEST_WIN sliding windows: input [i:i+L], target sessions[i+L]
    start_i = max(0, n - N_TEST_WIN - L_INPUT)
    for i in range(start_i, n - L_INPUT):
        ctx = sessions[i : i + L_INPUT]
        tgt = [t for t in sessions[i + L_INPUT] if t in svd_lookup]
        if not tgt:
            continue
        qvec = context_vec(ctx)
        if qvec is None:
            continue
        eval_queries.append((uid, qvec, set(tgt), ctx))
        if len(eval_queries) >= MAX_EVAL_QUERIES:
            break
    if len(eval_queries) >= MAX_EVAL_QUERIES:
        break

n_users_covered = len({q[0] for q in eval_queries})
print(f"Evaluation queries : {len(eval_queries):,}  from {n_users_covered:,} users  "
      f"({time.time()-t0:.1f}s)")
print(f"Avg GT tracks/query: "
      f"{np.mean([len(q[2]) for q in eval_queries]):.2f}")

## A5 — LSH Candidate Recall Sweep (Tables 1–10)

**Candidate Recall** = fraction of queries where ≥1 ground-truth track appears in the retrieved LSH buckets at all. This is an upper bound on Recall@10 for any downstream re-ranker.

Sweeping L=1..10 tables shows the recall vs. candidate-set-size tradeoff.

In [ ]:
table_sweep       = list(range(1, N_TABLES + 1))
cand_recall_list  = []
cand_size_list    = []

print("Sweeping L=1..10: candidate recall and avg candidate set size ...")
for n_t in table_sweep:
    hits = cand_sum = 0
    for _, qvec, gt_set, _ in eval_queries:
        cands     = set(lsh.query(qvec, n_tables=n_t))
        cand_sum += len(cands)
        if cands & gt_set:
            hits += 1
    cand_recall_list.append(hits / len(eval_queries))
    cand_size_list.append(cand_sum / len(eval_queries))
    print(f"  L={n_t:2d}  Candidate Recall={cand_recall_list[-1]:.3f}  "
          f"Avg candidates={cand_size_list[-1]:.0f}  "
          f"Reduction vs BF={100*(1-cand_size_list[-1]/UNIVERSE_SIZE):.1f}%")

## A6 — Recall@10 & NDCG@10: LSH+Cosine vs Brute-Force (Paper Protocol)

Metrics reported as percentages to match paper Table 1.  
**Brute-Force**: cosine over entire candidate universe.  
**LSH+Cosine**: cosine re-ranking restricted to LSH bucket contents.

In [ ]:
K          = K_TRACKS
EVAL_TABLES = [1, 3, 5, 7, 10]

# ── Brute-force baseline ─────────────────────────────────────────────────────
print("Computing brute-force cosine Recall@10 & NDCG@10 ...")
t0 = time.time()
bf_r_list, bf_n_list = [], []
for _, qvec, gt_set, _ in eval_queries:
    top = cosine_topk(qvec, session_tracks, K)
    bf_r_list.append(recall_at_k(top, gt_set, K))
    bf_n_list.append(ndcg_at_k(top,  gt_set, K))

bf_r10  = 100 * np.mean(bf_r_list)
bf_ndcg = 100 * np.mean(bf_n_list)
print(f"Brute-Force Cosine  Recall@10={bf_r10:.2f}%  NDCG@10={bf_ndcg:.2f}%  "
      f"({time.time()-t0:.1f}s)")

# ── LSH + cosine re-ranking at each L ────────────────────────────────────────
print("\nLSH + cosine re-ranking (L = tables used):")
lsh_results = {}
for n_t in EVAL_TABLES:
    r_list, n_list, cand_sum = [], [], 0
    for _, qvec, gt_set, _ in eval_queries:
        cands     = lsh.query(qvec, n_tables=n_t)
        cand_sum += len(cands)
        top       = cosine_topk(qvec, cands, K)
        r_list.append(recall_at_k(top, gt_set, K))
        n_list.append(ndcg_at_k(top,  gt_set, K))
    r10     = 100 * np.mean(r_list)
    ndcg    = 100 * np.mean(n_list)
    avg_c   = cand_sum / len(eval_queries)
    speedup = UNIVERSE_SIZE / avg_c if avg_c > 0 else float('inf')
    lsh_results[n_t] = {"R@10": r10, "NDCG@10": ndcg, "avg_cands": avg_c, "speedup": speedup}
    print(f"  L={n_t:2d}  Recall@10={r10:.2f}%  NDCG@10={ndcg:.2f}%  "
          f"candidates={avg_c:.0f}  speedup={speedup:.1f}×")

## A7 — LSH Tradeoff Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Candidate recall vs tables
axes[0].plot(table_sweep, [r*100 for r in cand_recall_list], 'b-o', markersize=6)
axes[0].axhline(100, color='gray', linestyle='--', label='Brute-force (100%)')
axes[0].set_title('Candidate Recall vs # Tables', fontweight='bold')
axes[0].set_xlabel('Number of LSH Tables (L)'); axes[0].set_ylabel('Candidate Recall (%)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Plot 2: Avg candidate set size
axes[1].plot(table_sweep, cand_size_list, 'r-o', markersize=6)
axes[1].axhline(UNIVERSE_SIZE, color='gray', linestyle='--', label=f'BF ({UNIVERSE_SIZE:,})')
axes[1].set_title('Avg Candidate Set Size vs # Tables', fontweight='bold')
axes[1].set_xlabel('Number of LSH Tables (L)'); axes[1].set_ylabel('Avg Candidates')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Plot 3: Recall@10 vs tables
lsh_r10_vals = [lsh_results[n_t]['R@10'] for n_t in EVAL_TABLES]
axes[2].plot(EVAL_TABLES, lsh_r10_vals, 'g-o', markersize=6, label='LSH+Cosine')
axes[2].axhline(bf_r10, color='gray', linestyle='--', label=f'BF Cosine ({bf_r10:.2f}%)')
axes[2].set_title('Recall@10 vs # Tables', fontweight='bold')
axes[2].set_xlabel('Number of LSH Tables (L)'); axes[2].set_ylabel('Recall@10 (%)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('LSH Tradeoff: Accuracy vs Computational Cost\n(Paper evaluation protocol)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CACHE_PATH}/lsh_tradeoff.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {CACHE_PATH}/lsh_tradeoff.png")

---
# PART B: MinHash Collaborative Filtering

**Theory**: Represent each user as set `S` of all tracks ever played. Jaccard similarity = `|A∩B|/|A∪B|`. MinHash: for each of k hash functions `h_i`, compute `min(h_i(t) for t in S)`. Probability that `min_h(A)==min_h(B)` equals Jaccard(A,B). Average over k → unbiased estimate.

**LSH banding fix**: Previous run used 20 bands × 5 rows → threshold ≈ 0.55. Music users have Jaccard ≈ 0.01–0.05. New: **50 bands × 2 rows → threshold ≈ 0.14**. Plus a vectorised direct-comparison fallback for users with very sparse overlap.

## B1 — MinHash Class

In [ ]:
class MinHash:
    """
    MinHash with universal hashing: h_i(x) = (a_i*x + b_i) mod p
    Signature: sig[i] = min over all items x in set of h_i(x)
    Jaccard estimate: mean(sig_A == sig_B)
    """
    def __init__(self, k: int = 100, seed: int = 42):
        self.k = k
        rng     = np.random.RandomState(seed)
        self.p  = (1 << 31) - 1                          # large prime
        self.a  = rng.randint(1, self.p, k).astype(np.int64)
        self.b  = rng.randint(0, self.p, k).astype(np.int64)

    def signature(self, item_set: set) -> np.ndarray:
        """Compute k-length MinHash signature for a set of integer item IDs."""
        if not item_set:
            return np.full(self.k, self.p, dtype=np.int64)
        items = np.array(list(item_set), dtype=np.int64) & 0x7FFFFFFF  # keep positive
        # hash_vals[i, j] = h_i(items[j])
        hash_vals = (np.outer(self.a, items) + self.b[:, None]) % self.p  # (k, |set|)
        return hash_vals.min(axis=1)                     # (k,)

    @staticmethod
    def jaccard_estimate(sig_a: np.ndarray, sig_b: np.ndarray) -> float:
        return float(np.mean(sig_a == sig_b))


class MinHashLSH:
    """
    LSH banding for MinHash signatures.
    Divide k-length signature into b bands of r rows.
    Two users are candidate similar if any band hash matches.
    Threshold Jaccard ≈ (1/b)^(1/r).
    """
    def __init__(self, k: int = 100, n_bands: int = 20, seed: int = 0):
        assert k % n_bands == 0, "k must be divisible by n_bands"
        self.k       = k
        self.b       = n_bands
        self.r       = k // n_bands        # rows per band
        self.buckets = [defaultdict(list) for _ in range(n_bands)]
        self.uid_list = []
        threshold = (1 / n_bands) ** (1 / self.r)
        print(f"MinHashLSH: k={k}, bands={n_bands}, rows/band={self.r}, "
              f"threshold Jaccard ≈ {threshold:.3f}")

    def build(self, uid_list: list, signatures: np.ndarray):
        """signatures: (N, k) int64 array."""
        self.uid_list = uid_list
        for idx, sig in enumerate(signatures):
            for band_id in range(self.b):
                start  = band_id * self.r
                band_h = tuple(sig[start : start + self.r].tolist())
                self.buckets[band_id][band_h].append(idx)
        print(f"MinHashLSH index built for {len(uid_list):,} users.")

    def candidates(self, sig: np.ndarray) -> set:
        """Return candidate user indices whose signature matches in at least one band."""
        cands = set()
        for band_id in range(self.b):
            start  = band_id * self.r
            band_h = tuple(sig[start : start + self.r].tolist())
            cands.update(self.buckets[band_id].get(band_h, []))
        return cands


print("MinHash and MinHashLSH classes defined.")

MinHash and MinHashLSH classes defined.


## B2 — Build User Track Sets & MinHash Signatures

In [ ]:
K_HASH = 100
mh     = MinHash(k=K_HASH)

print(f"Building MinHash signatures for {len(user_sessions):,} users ...")
t0 = time.time()

uid_list   = list(user_sessions.keys())
signatures = np.zeros((len(uid_list), K_HASH), dtype=np.int64)
user_sets  = {}    # uid → set of all unique tracks (reused in CF recommendation)

for i, uid in enumerate(uid_list):
    track_set      = set(t for sess in user_sessions[uid] for t in sess)
    user_sets[uid] = track_set
    signatures[i]  = mh.signature(track_set)
    if (i + 1) % 20000 == 0:
        print(f"  {i+1:,}/{len(uid_list):,}", end='\r')

uid_to_idx = {uid: i for i, uid in enumerate(uid_list)}
print(f"\nDone. Signatures: {signatures.shape}  ({time.time()-t0:.1f}s)")

# MinHash accuracy check on 500 random pairs
sample_pairs = np.random.choice(len(uid_list), (500, 2), replace=False)
errors = []
for i, j in sample_pairs:
    uid_i, uid_j = uid_list[i], uid_list[j]
    s_i, s_j     = user_sets[uid_i], user_sets[uid_j]
    exact_jac    = len(s_i & s_j) / len(s_i | s_j) if s_i | s_j else 0.0
    mh_est       = mh.jaccard_estimate(signatures[i], signatures[j])
    errors.append(abs(exact_jac - mh_est))

print(f"\nMinHash accuracy (500 pairs): mean|error|={np.mean(errors):.4f}  "
      f"max={np.max(errors):.4f}  (theory σ={1/K_HASH**0.5:.4f})")

# Actual Jaccard distribution sample
sample_jac = [len(user_sets[uid_list[i]] & user_sets[uid_list[j]]) /
              len(user_sets[uid_list[i]] | user_sets[uid_list[j]])
              for i, j in sample_pairs if user_sets[uid_list[i]] | user_sets[uid_list[j]]]
print(f"Observed Jaccard: mean={np.mean(sample_jac):.4f}  "
      f"p50={np.percentile(sample_jac,50):.4f}  "
      f"p90={np.percentile(sample_jac,90):.4f}  "
      f"max={np.max(sample_jac):.4f}")

## B3 — Build MinHash LSH Index (Fixed Banding: N_BANDS=50)

**Key fix from previous run**: N_BANDS=20, r=5 gave threshold ≈ 0.549. Music users have Jaccard ≈ 0.01–0.05, so 98.6% of queries found zero candidates.

**Fix**: N_BANDS=50, r=2 → threshold ≈ (1/50)^(1/2) = **0.141**. Still high but dramatically better.

**Additional fix**: Vectorised direct-comparison fallback — if LSH banding finds <20 candidates, directly scan all users with numpy (`jac_est = mean(sigs == query_sig, axis=1)`) and keep any with estimate ≥ 0.04. O(N×k) = O(50k×100) = 5M ops, runs in milliseconds.

In [ ]:
N_BANDS = 50   # 50 bands × 2 rows → threshold Jaccard ≈ (1/50)^(1/2) = 0.141
               # (previous: 20 bands × 5 rows → 0.549 — too high for music Jaccard ≈ 0.01–0.05)

mhlsh = MinHashLSH(k=K_HASH, n_bands=N_BANDS)
mhlsh.build(uid_list, signatures)

print("\nProbability of being a candidate pair at different Jaccard values:")
for s in [0.02, 0.04, 0.08, 0.14, 0.20, 0.30, 0.50]:
    r    = K_HASH // N_BANDS
    prob = 1 - (1 - s**r)**N_BANDS
    print(f"  Jaccard={s:.2f}  →  P(candidate) = {prob:.3f}")

## B4 — Generate CF Recommendations & Evaluate R@10

In [ ]:
MIN_JACCARD = 0.04   # fallback threshold for direct comparison

def minhash_recommend(test_uid, context_sessions, top_k=10):
    """
    1. Build track set from context sessions.
    2. Compute MinHash signature.
    3. LSH banding → candidate users.
    4. Fallback: if <20 candidates, vectorised direct scan (O(N×k), fast with numpy).
    5. Aggregate weighted track votes from top-50 most similar users.
    """
    train_set = set(t for sess in context_sessions for t in sess)
    test_sig  = mh.signature(train_set)

    # LSH banding (fast, approximate)
    cand_idxs = mhlsh.candidates(test_sig)
    cand_idxs.discard(uid_to_idx.get(test_uid))

    # Vectorised fallback: scan all users, keep those with MinHash Jaccard ≥ threshold
    if len(cand_idxs) < 20:
        jac_all  = np.mean(signatures == test_sig[None, :], axis=1)   # (N,)
        fallback = set(int(i) for i in np.where(jac_all >= MIN_JACCARD)[0])
        fallback.discard(uid_to_idx.get(test_uid))
        cand_idxs = cand_idxs | fallback

    if not cand_idxs:
        return [], 0

    cand_list = list(cand_idxs)
    cand_sigs = signatures[cand_list]
    jac_ests  = np.mean(cand_sigs == test_sig[None, :], axis=1)

    top_idxs  = np.argsort(jac_ests)[::-1][:50]

    track_votes = defaultdict(float)
    for ri in top_idxs:
        other_uid = uid_list[cand_list[ri]]
        sim       = float(jac_ests[ri])
        if sim <= 0:
            continue
        for t in (user_sets[other_uid] - train_set):
            track_votes[t] += sim

    if not track_votes:
        return [], len(cand_idxs)
    return sorted(track_votes, key=track_votes.get, reverse=True)[:top_k], len(cand_idxs)


print(f"Evaluating MinHash CF Recall@10 & NDCG@10 on {len(eval_queries):,} queries ...")
t0 = time.time()

mh_r_list, mh_n_list = [], []
mh_cand_sum = no_cand = 0

for uid, qvec, gt_set, ctx in eval_queries:
    recs, n_cands  = minhash_recommend(uid, ctx, top_k=K)
    mh_cand_sum   += n_cands
    if not recs:
        no_cand += 1
    mh_r_list.append(recall_at_k(recs, gt_set, K))
    mh_n_list.append(ndcg_at_k(recs,  gt_set, K))

mh_r10  = 100 * np.mean(mh_r_list)
mh_ndcg = 100 * np.mean(mh_n_list)
print(f"MinHash CF  Recall@10={mh_r10:.2f}%  NDCG@10={mh_ndcg:.2f}%")
print(f"Avg candidates/query: {mh_cand_sum/len(eval_queries):.1f}")
print(f"Queries with zero candidates: {no_cand}/{len(eval_queries)} "
      f"({100*no_cand/len(eval_queries):.1f}%)")
print(f"Elapsed: {time.time()-t0:.1f}s")

## B5 — Final Comparison: Our Methods vs REACTA Paper Baselines (Table 1)

In [ ]:
best_l   = max(lsh_results, key=lambda l: lsh_results[l]['R@10'])
best_lsh = lsh_results[best_l]

W = 72
print("=" * W)
print(f"{'RESULTS vs REACTA PAPER BASELINES (Table 1, RecSys 2025)':^{W}}")
print("=" * W)
print(f"\n{'Method':<28}  {'Recall@10':>10}  {'NDCG@10':>10}  Notes")
print("-" * W)

# Paper baselines
for name, vals in PAPER_BASELINES.items():
    marker = " ← best" if name == "REACTA-P" else ""
    print(f"{name:<28}  {vals['Recall@10']:>9.2f}%  {vals['NDCG@10']:>9.2f}%{marker}")

print("-" * W)

# Our methods
our_results = [
    ("Brute-Force Cosine",
     bf_r10, bf_ndcg,
     f"cosine over {UNIVERSE_SIZE:,} tracks"),
    (f"LSH+Cosine (L={best_l})",
     best_lsh['R@10'], best_lsh['NDCG@10'],
     f"{best_lsh['avg_cands']:.0f} candidates, {best_lsh['speedup']:.0f}× faster"),
    ("MinHash CF (fixed)",
     mh_r10, mh_ndcg,
     "user-based CF via Jaccard similarity"),
]
for name, r10, ndcg, note in our_results:
    print(f"{name:<28}  {r10:>9.2f}%  {ndcg:>9.2f}%  {note}")

print("=" * W)

# Summary
best_paper_r10  = PAPER_BASELINES['REACTA-P']['Recall@10']
best_paper_ndcg = PAPER_BASELINES['REACTA-P']['NDCG@10']
print(f"\nBest paper   (REACTA-P):   Recall@10={best_paper_r10:.2f}%  NDCG@10={best_paper_ndcg:.2f}%")
print(f"Our best LSH (L={best_l}): "
      f"Recall@10={best_lsh['R@10']:.2f}%  NDCG@10={best_lsh['NDCG@10']:.2f}%")
gap_r = best_lsh['R@10'] - best_paper_r10
gap_n = best_lsh['NDCG@10'] - best_paper_ndcg
print(f"Gap: Recall@10={gap_r:+.2f}pp  NDCG@10={gap_n:+.2f}pp")
print(f"\nLSH efficiency: L={best_l} uses {best_lsh['avg_cands']:.0f} candidates vs {UNIVERSE_SIZE:,} BF "
      f"({best_lsh['speedup']:.0f}× speedup) retaining "
      f"{100*best_lsh['R@10']/bf_r10:.1f}% of brute-force accuracy")

## Final Summary Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

paper_names  = list(PAPER_BASELINES.keys())
paper_r10s   = [PAPER_BASELINES[n]['Recall@10']  for n in paper_names]
paper_ndcgs  = [PAPER_BASELINES[n]['NDCG@10']    for n in paper_names]
our_names    = ["BF\nCosine", f"LSH\nL={best_l}", "MinHash\nCF"]
our_r10s     = [bf_r10, best_lsh['R@10'], mh_r10]
our_ndcgs    = [bf_ndcg, best_lsh['NDCG@10'], mh_ndcg]

all_names = paper_names + our_names
all_r10s  = paper_r10s  + our_r10s
all_ndcgs = paper_ndcgs + our_ndcgs
colors    = ['#9ecae1'] * len(paper_names) + ['steelblue', 'seagreen', 'coral']

for ax_idx, (ax, vals, ylabel, title) in enumerate(zip(
        axes,
        [all_r10s, all_ndcgs],
        ["Recall@10 (%)", "NDCG@10 (%)"],
        ["Recall@10 vs Paper Baselines", "NDCG@10 vs Paper Baselines"])):
    bars = ax.bar(range(len(all_names)), vals, color=colors, alpha=0.88, width=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.08,
                f"{v:.2f}%", ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(all_names)))
    ax.set_xticklabels(all_names, rotation=20, ha='right', fontsize=9)
    # Divider between paper and ours
    ax.axvline(len(paper_names) - 0.5, color='black', linestyle='--', alpha=0.4, lw=1.5)
    ax.text(len(paper_names)/2 - 0.5, max(vals)*1.02, "Paper Baselines",
            ha='center', fontsize=9, color='#555', fontstyle='italic')
    ax.text(len(paper_names) + len(our_names)/2 - 0.5, max(vals)*1.02, "Our Methods",
            ha='center', fontsize=9, color='#555', fontstyle='italic')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(vals) * 1.18)

plt.suptitle("LSH + MinHash CF vs REACTA Paper Baselines\n"
             f"(Full dataset, L={L_INPUT} sessions input → predict session {L_INPUT+1}, K={K_TRACKS})",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CACHE_PATH}/paper_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {CACHE_PATH}/paper_comparison.png")